# Digit Recognizer (ConvNet Only)

by **Dohyeop Lim**

In [ ]:
import os
from datetime import datetime
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
from dataclasses import dataclass
from typing import Optional, Dict, List
from tqdm.auto import tqdm

## Prep

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
train_df = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")
test_df = pd.read_csv("/kaggle/input/digit-recognizer/test.csv")

In [ ]:
print(train_df.shape)
print(test_df.shape)

In [ ]:
train_df.describe()

### Data Overview

- Image size: 784 pixels = 28 * 28
- Each pixel: 0.0 - 254.0
- Label: column `label`, 0 - 9

## Datasets & Loaders

reference: [PyTorch Docs](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html#creating-a-custom-dataset-for-your-files)

In [ ]:
class Digit(Dataset):
    def __init__(self, df: pd.DataFrame, transform: transforms.Compose, isTrain=True):
        self.df = df.copy()
        self.transform = transform
        self.isTrain = isTrain
        
        if isTrain:
            self.labels = self.df.iloc[:, 0].values
            self.images = self.df.iloc[:, 1:].values.reshape(-1, 28, 28)
        else:
            self.images = self.df.values.reshape(-1, 28, 28)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.fromarray(self.images[idx].astype(np.uint8))
        if self.transform:
            img = self.transform(img)
        if self.isTrain:
            label = int(self.labels[idx])
            return img, label
        else:
            return img

- **Data augmentation**
    - Shift about pm 3px
    - Scale 0.9x-1.1x
    - Brightness 0.9-1.1
    - Contrast 0.8-1.2

In [ ]:
SHIFT = 3/28

aug = transforms.Compose([
    transforms.RandomAffine(degrees=0, translate=(SHIFT, SHIFT), scale=(0.9, 1.1), fill=0),
    transforms.ColorJitter(brightness=0.1, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,),(0.3081,))
])

norm_only = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,),(0.3081,))
])

In [ ]:
VAL_RATIO = 0.1

labels = train_df.iloc[:, 0]
train_df, val_df = train_test_split(train_df, test_size=VAL_RATIO, random_state=42, stratify=labels)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

In [ ]:
train_dataset = Digit(train_df, transform=aug)
val_dataset = Digit(val_df, transform=norm_only)
test_dataset = Digit(test_df, transform=norm_only, isTrain=False)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

## ConvNet Model

In [ ]:
class ConvNet1(nn.Module):
    def __init__(self):
        super().__init__()

        self.extractor = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.extractor(x)
        x = self.classifier(x)
        return x

## Train & Eval

In [ ]:
@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        total_loss += loss.item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return {
        "loss": total_loss / total,
        "acc": correct / total
    }

In [ ]:
@dataclass
class TrainConfig:
    epochs = 20
    use_amp = True
    grad_clip: Optional[float] = 1.0
    patience = 5
    scheduler_step_per_batch = False

def train(model, train_loader, val_loader, optimizer, scheduler=None, config: TrainConfig=TrainConfig()):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    scaler = GradScaler("cuda", enabled=(config.use_amp and device.type == "cuda"))

    best_state: Dict[str, torch.Tensor] = {}
    best_acc = -1.0
    no_improve = 0
    history: List[Dict] = []

    for epoch in range(1, config.epochs + 1):
        model.train()
        running_loss, running_correct, running_total = 0.0, 0, 0
        
        progress = tqdm(train_loader, desc=f"Epoch {epoch}/{config.epochs}", leave=False)
        for x, y in progress:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)

            with autocast("cuda", enabled=scaler.is_enabled()):
                logits = model(x)
                loss = F.cross_entropy(logits, y)

            if scaler.is_enabled():
                scaler.scale(loss).backward()
                if config.grad_clip:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                if config.grad_clip:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
                optimizer.step()

            if scheduler and config.scheduler_step_per_batch:
                scheduler.step()

            bs = y.size(0)
            running_loss += loss.item() * bs
            running_correct += (logits.argmax(1) == y).sum().item()
            running_total += bs

            progress.set_postfix(
                train_loss = running_loss / running_total,
                train_acc = running_correct / running_total
            )

        if scheduler and not config.scheduler_step_per_batch:
            scheduler.step()

        val_metrics = validate(model, val_loader, device)
        train_loss = running_loss / running_total
        train_acc = running_correct / running_total

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
            "lr": (scheduler.get_last_lr()[0] if scheduler else optimizer.param_groups[0]["lr"])
        })

        if val_metrics["acc"] > best_acc:
            best_acc = val_metrics["acc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= config.patience:
                break

        print(f"Epoch {epoch:02d} | "
              f"train {train_loss:.4f}/{train_acc:.4f} | "
              f"val {val_metrics['loss']:.4f}/{val_metrics['acc']:.4f} | "
              f"lr {(scheduler.get_last_lr()[0] if scheduler else optimizer.param_groups[0]['lr']):.2e}")

    if best_state:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    return model, history, best_acc

In [ ]:
model = ConvNet1()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

config = TrainConfig()
model, history, best_acc = train(model, train_loader, val_loader, optimizer, scheduler, config)
print("Best val acc:", best_acc)

## Test

In [ ]:
@torch.no_grad()
def predict(model, loader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    preds = []
    for x in loader:
        x = x.to(device)
        logits = model(x)
        preds.append(logits.argmax(1).cpu().numpy())
    return np.concatenate(preds)

In [ ]:
preds = predict(model, test_loader)

In [ ]:
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
filename = f"/kaggle/working/submission_{timestamp}.csv"

pd.DataFrame({
    "ImageId": np.arange(1, len(preds) + 1),
    "Label": preds
}).to_csv(filename, index=False)

print(filename)